# Ferrioxalate Actinometry

Complete workflow for ferrioxalate actinometry. Run the cells top to bottom.

1. **Setup** — imports and shared functions
2. **Part 1** — Irradiation time calculator (before experiment)
4. **Part 2** — Photon flux from UV-Vis absorbance measurement (after experiment)

---
## Setup — Imports & Shared Functions
Run this cell first.

In [40]:
import numpy as np
import pandas as pd
from scipy import constants

# ---------------------------------------------------------------------------
# Photodiode calibration data and correction factor interpolation
# ---------------------------------------------------------------------------

CALIBRATION_DATA = {
    "wavelength": [
        260,270,280,290,300,310,320,330,340,350,360,370,380,390,400,
        410,420,430,440,450,460,470,480,490,500,510,520,530,540,550,
        560,570,580,590,600,610,620,630,640,650,660
    ],
    "high": [
        10.042,9.892,9.493,8.934,8.301,7.654,7.036,6.467,5.956,5.503,
        5.105,4.758,4.454,4.189,3.958,3.756,3.578,3.423,3.285,3.164,
        3.056,2.960,2.874,2.797,2.727,2.663,2.605,2.552,2.502,2.455,
        2.410,2.367,2.325,2.284,2.244,2.203,2.162,2.121,2.080,2.038,
        1.995
    ],
    "low": [
        0.502,0.495,0.475,0.447,0.415,0.383,0.352,0.323,0.298,0.275,
        0.255,0.238,0.223,0.209,0.198,0.188,0.179,0.171,0.164,0.158,
        0.153,0.148,0.144,0.140,0.136,0.133,0.130,0.128,0.125,0.123,
        0.120,0.118,0.116,0.114,0.112,0.110,0.108,0.106,0.104,0.102,
        0.100
    ]
}

def get_correction_factor(wavelength, channel='low'):
    """Interpolate photodiode correction factor [mW/V]."""
    return np.interp(wavelength, CALIBRATION_DATA['wavelength'], CALIBRATION_DATA[channel])


# ---------------------------------------------------------------------------
# UV-Vis CSV parser
# ---------------------------------------------------------------------------

def produce_UV_Vis_dataframe(file):
    df = pd.read_csv(file, skip_blank_lines=False)
    empty_row_mask = df.replace("", np.nan).isna().all(axis=1)
    if empty_row_mask.any():
        last_row = empty_row_mask[empty_row_mask].index[0]
        df = df.iloc[1:last_row]
    df.columns = [None] + list(df.columns[:-1])
    cols_to_drop = [i for i in range(1, len(df.columns)) if i % 2 == 0]
    df = df.drop(columns=df.columns[cols_to_drop])
    return df.astype(float)

def extract_absorption_at_wavelength(dataframe, wavelength=510):
    idx = (dataframe.iloc[:, 0] - wavelength).abs().idxmin()
    return dataframe.iloc[idx, 1:].tolist()


# ---------------------------------------------------------------------------
# QYDS .dat file parser
# ---------------------------------------------------------------------------

def produce_QYDS_dataframe(file):
    data = {'Runtime': [], 'Illum Time': [], 'Cell Value': []}
    with open(file, 'r') as f:
        lines = f.readlines()
    for line in lines:
        if line.startswith('#') or line.strip().startswith('Number'):
            continue
        parts = line.split('\t')
        if len(parts) >= 7:
            try:
                data['Runtime'].append(float(parts[3].replace(',', '.')))
                data['Illum Time'].append(float(parts[4].replace(',', '.')))
                data['Cell Value'].append(float(parts[6].replace(',', '.')))
            except ValueError:
                continue
    return pd.DataFrame(data).astype(float)


print('Setup complete.')

Setup complete.


---
## Part 1 — Irradiation Time Calculator

Run **before** your experiment to estimate the required irradiation time.

Edit the parameters in the block below, then run the cell.

In [41]:
# ── Set your parameters here ───────────────────────────────────────────────
wavelength         = 365    # nm
channel            = 'low'  # 'low' or 'high'
voltage            = 3.51   # V  (measured voltage after reference)
desired_absorbance = 0.5    # target absorbance at 510 nm
# ──────────────────────────────────────────────────────────────────────────

cf          = get_correction_factor(wavelength, channel)
estimated_photon_flux = wavelength * voltage * cf / (constants.Planck * constants.speed_of_light * constants.Avogadro) * 1e-12
result_time = (desired_absorbance * 3 * 10) / (1.20 * 11100 * 1 * 1 * photon_flux) * 1e-3

print('─' * 52)
print('INPUT PARAMETERS')
print(f'  Wavelength              : {wavelength} nm')
print(f'  Channel                 : {channel}')
print(f'  Voltage after reference : {voltage} V')
print(f'  Desired absorbance      : {desired_absorbance}')
print('─' * 52)
print('RESULTS')
print(f'  Correction factor       : {cf:.4f} mW V\u207b\u00b9')
print(f'  Estimated photon flux   : {estimated_photon_flux:.2e} mol s\u207b\u00b9')
print(f'  Required irradiation    : {result_time:.0f} s  ({result_time/60:.1f} min)')
print('─' * 52)

────────────────────────────────────────────────────
INPUT PARAMETERS
  Wavelength              : 365 nm
  Channel                 : low
  Voltage after reference : 3.51 V
  Desired absorbance      : 0.5
────────────────────────────────────────────────────
RESULTS
  Correction factor       : 0.2465 mW V⁻¹
  Estimated photon flux   : 2.64e-09 mol s⁻¹
  Required irradiation    : 420 s  (7.0 min)
────────────────────────────────────────────────────


---
## Part 2 — Photon Flux from UV-Vis Measurement

Use **after** your experiment. The CSV should contain the blank spectrum (column 1) and the irradiated spectrum (column 2). ΔA at 510 nm is used to back-calculate the photon flux.

**Supported wavelengths and actinometry settings:**

| λ / nm | Concentration / mM | Quantum yield |
|---|---|---|
| 254 – 366 | 6 | 1.23 |
| 405 | 6 | 1.14 |
| 436 | 6 | 1.11 |
| 458 | 10 | 1.12 |
| 488 | 80 | 1.08 |
| 514 | 200 | 0.93 |

Edit the parameters below, then run the cell.

In [42]:
# ── Set your parameters here ───────────────────────────────────────────────
uv_file          = r"C:\Users\metzen\sciebo\Promotion\10_Scripts\00_Github\QYDS_Quantum_Yield_Calculator\testfiles\Actinometry\251205_Ferrioxalate.csv" # path to UV-Vis CSV file
wavelength       = 365   # nm  (irradiation wavelength)
irradiation_time = 400   # s
# ──────────────────────────────────────────────────────────────────────────

# Fixed instrumental / experimental constants
V1, V2, V3 = 3, 1, 10   # mL: irradiated, transferred, total analytic volume
eps, d     = 11100, 1    # L mol-1 cm-1, cm

WAVELENGTH_SETTINGS = {
    405: (6, 1.14), 436: (6, 1.11), 458: (10, 1.12),
    488: (80, 1.08), 514: (200, 0.93)
}

if 254 <= wavelength <= 366:
    conc, QY = 6, 1.23
elif wavelength in WAVELENGTH_SETTINGS:
    conc, QY = WAVELENGTH_SETTINGS[wavelength]
else:
    raise ValueError(f'No actinometry settings defined for {wavelength} nm.')

df          = produce_UV_Vis_dataframe(uv_file)
absorbances = extract_absorption_at_wavelength(df, wavelength=510)
dA          = absorbances[1] - absorbances[0]
photon_flux = dA * V1 * V3 / (QY * eps * V2 * d * irradiation_time) * 1e-3

print('─' * 55)
print('INPUT PARAMETERS')
print(f'  Irradiation wavelength        : {wavelength} nm')
print(f'  Channel                       : {channel}')
print(f'  Voltage after reference       : {voltage} V')
print(f'  Irradiation time              : {irradiation_time} s')
print('USED PARAMETERS')
print(f'  Concentration                 : {conc} mM')
print(f'  Quantum yield                 : {QY}')
print(f'  V\u2081 (irradiated)               : {V1} mL')
print(f'  V\u2082 (transferred)              : {V2} mL')
print(f'  V\u2083 (analytic total)           : {V3} mL')
print(f'  \u03b5 (510 nm)                    : {eps} L mol\u207b\u00b9 cm\u207b\u00b9')
print(f'  Path length                   : {d} cm')
print('─' * 55)
print('RESULTS')
print(f'  A (blank)                     : {absorbances[0]:.4f}')
print(f'  A (irradiated)                : {absorbances[1]:.4f}')
print(f'  \u0394A at 510 nm                  : {dA:.4f}')
print(f'  Photon flux from actionometry : {photon_flux:.2e} mol s\u207b\u00b9')
print(f'  Estimated photon flux         : {estimated_photon_flux:.2e} mol s\u207b\u00b9')
print('─' * 55)

───────────────────────────────────────────────────────
INPUT PARAMETERS
  Irradiation wavelength        : 365 nm
  Channel                       : low
  Voltage after reference       : 3.51 V
  Irradiation time              : 400 s
USED PARAMETERS
  Concentration                 : 6 mM
  Quantum yield                 : 1.23
  V₁ (irradiated)               : 3 mL
  V₂ (transferred)              : 1 mL
  V₃ (analytic total)           : 10 mL
  ε (510 nm)                    : 11100 L mol⁻¹ cm⁻¹
  Path length                   : 1 cm
───────────────────────────────────────────────────────
RESULTS
  A (blank)                     : 0.3497
  A (irradiated)                : 0.8382
  ΔA at 510 nm                  : 0.4885
  Photon flux from actionometry : 2.68e-09 mol s⁻¹
  Estimated photon flux         : 2.64e-09 mol s⁻¹
───────────────────────────────────────────────────────
